# Example run of TIPSY and MCMC: S CrA

For this, we are using the ALMA 13 cube of S CrA, offered in the original version of TIPSY from <a href="https://virginia.app.box.com/s/4xwhmdh5gvsapybt5jurs5ey22i4kiuq"> here</a>

Steps 1 and 2 are exactly the same as the original TIPSY workflow.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from astropy.wcs import WCS
from astropy.io import fits
from astropy.coordinates import SkyCoord
from astropy import units as u
from spectral_cube import SpectralCube
from scipy.optimize import minimize
from uncertainties import ufloat, unumpy, umath

import sys
sys.path.append('/Users/mvaldivi/Documents/Science/TIPSY/')
from trivia_my import make_ppv
from tipsy import *
from tipsy_mcmc import *

import pickle
import emcee
import corner

## Step 1: read fits file and isolate streamer

In [1]:
source = 'SCrA'

## Stellar properties
Ms_val = 1+1 #Mass in solar mass
dist = 160 *u.pc #Distance with units
svel = 5.86 #Systemic velocity in km/s

fitsfolder = ''
fitsfile = fitsfolder + source + '_13CO_spw25_tav_jupyter_shifted.fits'
molecule = '13CO'
N_elements = 10  # Number of elements to consider in the trajectory fit

NameError: name 'u' is not defined

In [ ]:

hdu = fits.open(fitsfile)[0]   # Reading file 
width = min(hdu.header['CDELT2']*(hdu.header['NAXIS2']//2),
            -(hdu.header['CDELT1']*(hdu.header['NAXIS1']//2)))*60*60 # width of the cube, used for visualisation
cube_init = SpectralCube.read(hdu).with_spectral_unit(u.km/u.s, velocity_convention='radio',rest_value=hdu.header['RESTFRQ']*u.Hz)
scube = cube_init # Cube as SpectralCube object

## Plotting whole cube, just to see where the streamer is
make_ppv(fitsfile,show_figure = True,projection_z = True,write_pdf = False,write_html = True
         ,clip =4,rmax=1*width,vunit_per_s = 'm',vmin=-5,vmax=20
         ,bool_traj = False)

In [ ]:
## Limits for extracting subcube with streamer
vmin_s = 7    # Min. vel. of streamer
vmax_s = 10
xmin_s = -3   # Min R.A. offset to consider for streamer
xmax_s = -1
ymin_s = -3   # Min. Decl. offset to consider for stremar
ymax_s = 0.5
rms_thresh = 4  # sigma threshold for streamer


## Extracting subcube with streamer
scubevc2 = streamer_subcube(scube,xmin_s,xmax_s,ymin_s,ymax_s,vmin_s,vmax_s,rms_thresh=rms_thresh)

scubevc2.write(fitsfile[:-5]+'_streamer-1.fits',format = 'fits',overwrite = True) # saving streamer subcube for PPV
make_ppv(fitsfile[:-5]+'_streamer-1.fits',show_figure = True,projection_z = False,write_pdf = False,write_html = False
         ,clip =0,rmax = width*1,vunit_per_s = 'km'
         ,bool_traj = False)
os.system('rm '+fitsfile[:-5]+'_streamer-1.fits') # Deleting the saved .fits file

In [ ]:
## Using clustering algorithm to remove random blobs of emission in the same subcube (optional)
scubevc3 = streamer_cleaning(scubevc2,model_args = {'xi':0.25})
print("Final streamer to fit:")
scubevc3.write(fitsfile[:-5]+'_streamer.fits',format = 'fits',overwrite = True)  # saving isolated streamer for future reference

## Step 2: initial fit with TIPSY

In [ ]:
params,bestTraj,others = traj_fitting(scubevc3,Ms_val,dist,svel, N_elements=N_elements,
                            ,z0_lim = (-2500,500),z0_step=50 # Setting parameter space, optional
                            ,vxy0_lim = (0,5) # Setting parameter space, optional
                            ,fit2html=False,html_path=fitsfile[:-5]+'_3Dfit.html')

## To store results for future use
results = [params,bestTraj,others]
with open(fitsfile[:-5]+'_fitresults.pkl', 'wb') as f:
    pickle.dump(results, f)

## Step 3: first maximum likelihood fit

In [ ]:
# Read TIPSY results
with open(fitsfile[:-5]+'_fitresults.pkl', 'rb') as f:
    results = pickle.load(f)
params,bestTraj,others = results 

perrs = parameter_errors(params,threshold = 0.90)
pccoords, flux = cube_to_pointcloud(scubevc3)
d_means, x_t, y_t, vz_t, x_err, y_err, vz_err = transform_cube_into_ydata(scubevc3, N_elements=N_elements)
data_array = np.array([x_t, y_t, vz_t])
error_array = np.array([x_err, y_err, vz_err])
[x0_as, y0_as, vz0], [x0_as_err, y0_as_err, vz0_err] = find_initial_conditions(data_array[0], data_array[1], data_array[2], 
                                                                               error_array[0], error_array[1], error_array[2], svel)
x0 = arcsec2au(x0_as, dist) # au
y0 = arcsec2au(y0_as, dist)
x0_err = arcsec2au(x0_as_err, dist) # au
y0_err = arcsec2au(y0_as_err, dist)
# initial conditions based on TIPSY results
z0, vx0, vy0 = perrs.loc['z0', 'value'], perrs.loc['vx0', 'value'], perrs.loc['vy0', 'value']
#z0, vxy0, vxy_ang0 = perrs.loc['z0', 'value'], perrs.loc['vxy0', 'value'], perrs.loc['vxy_ang0', 'value']
theta = [z0, vx0, vy0]
#theta = [z0, vxy0, vxy_ang0]
params_fixed = [x0, y0, vz0, Ms_val, dist, svel]



In [ ]:
# Always define your random seed!
np.random.seed(42)
# Log likelihood is a function in TIPSY_MCMC
nll = lambda x: -log_likelihood(x, d_means=d_means, y=data_array, yerr=error_array, 
                                params_fixed=params_fixed, pccoords=pccoords)
initial = theta
soln = minimize(nll, initial, method='Nelder-Mead') # test different methods between TNC, Nelder-Mead or Powell

if soln.success:
    fitted_params = soln.x
    print("Optimized parameters:")
    print(f"z0 = {fitted_params[0]:.2f} au")
    print(f"vx0 = {fitted_params[1]:.4f} km/s")
    print(f"vy0 = {fitted_params[2]:.4f} km/s")
    print('Original TIPSY parameters:')
    print(f"z0 = {perrs.loc['z0', 'value']:.2f} au")
    print(f"vx0 = {perrs.loc['vx0', 'value']:.4f} km/s")
    print(f"vy0 = {perrs.loc['vy0', 'value']:.4f} km/s")
else:
    print("Optimization failed.")


## Step 4: parameter exploration and error estimation using MCMC

In [ ]:
perrs

In [ ]:
# setup MCMC and backend
nwalkers = 100
ndim = len(theta)
filename_back = source + '_'+ molecule+ "_mcmcchain.h5"
labels = ["z0", "vx0", "vy0"]

# define the posterior function
# the definition of the prior is important for the convergence due to the angle variation. 
# Use the information obtained in the first TIPSY fit.
# A small value of vxy0 can make the angle diverge (and meaningless), so its best to limit vx0 and vy0 (absolute values).

def log_prior(theta):
    '''
    I think the priors should be set in a case-by-case basis depending 
    on the source and data.
    '''
    z0, vx0, vy0 = theta
    max_vel_range = 3 * np.max([perrs.loc['vx0', 'error'], perrs.loc['vy0', 'error']])
    max_space_range = 3 * perrs.loc['z0', 'error']
    if (-max_space_range <= z0 <= max_space_range) and (perrs.loc['vx0', 'value']-max_vel_range < vx0 < perrs.loc['vx0', 'value']+max_vel_range) and (perrs.loc['vy0', 'value']-max_vel_range < vy0 < perrs.loc['vy0', 'value']+max_vel_range):
    # if (-250 <= z0 <= 2000) and (0 < vxy0 < 1) and (perrs.loc['vxy_ang0', 'value']-2*perrs.loc['vxy_ang0', 'error'] < vxy_ang0 < perrs.loc['vxy_ang0', 'value']+2*perrs.loc['vxy_ang0', 'error']): 
        return 0.0
    return -np.inf

def log_probability(theta, d_means, y, yerr, params_fixed, pccoords, theta_weight=1.0):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    # print(lp)
    # print(log_likelihood(theta, d_means, y, yerr, params_fixed, pccords))
    return lp + log_likelihood(theta, d_means, y, yerr, params_fixed, pccoords)

### Running and saving into backend

In [ ]:
# Set up the backend
# Don't forget to clear it in case the file already exists
backend = emcee.backends.HDFBackend(filename_back)
backend.reset(nwalkers, ndim)
# we define a tiny Gaussian ball around the optimized solution so that the walkers can start
pos = soln.x + 1e-4 * np.random.randn(nwalkers, ndim)

sampler = emcee.EnsembleSampler(
    nwalkers, ndim, log_probability, 
    args=(d_means, data_array, error_array, params_fixed, pccoords),
    backend=backend
)


In [ ]:
max_n = 10000

# We'll track how the average autocorrelation time estimate changes
index = 0
autocorr = np.empty(max_n)

# This will be useful to testing convergence
old_tau = np.inf

# Now we'll sample for up to max_n steps
for sample in sampler.sample(pos, iterations=max_n, progress=True):
    # Only check convergence every 100 steps
    if sampler.iteration % 100:
        continue

    # Compute the autocorrelation time so far
    # Using tol=0 means that we'll always get an estimate even
    # if it isn't trustworthy
    tau = sampler.get_autocorr_time(tol=0)
    autocorr[index] = np.mean(tau)
    index += 1

    # Check convergence
    converged = np.all(tau * 100 < sampler.iteration)
    converged &= np.all(np.abs(old_tau - tau) / tau < 0.01)
    if converged:
        break
    old_tau = tau

In [ ]:
fig, axes = plt.subplots(3, figsize=(10, 7), sharex=True)
samples = sampler.get_chain()

for i in range(ndim):
    ax = axes[i]
    ax.plot(samples[:, :, i], "k", alpha=0.3)
    ax.set_xlim(0, len(samples))
    ax.set_ylabel(labels[i])
    ax.yaxis.set_label_coords(-0.1, 0.5)

axes[-1].set_xlabel("step number")
fig.savefig(source + '_'+ molecule+ "_mcmc_walkers.pdf", dpi=72)

### Reloading (not to run it again as it takes time)

In [ ]:
backend = emcee.backends.HDFBackend(filename_back)


In [ ]:
tau = backend.get_autocorr_time()
print(tau)

flat_samples = backend.get_chain(discard=int(tau[0]), flat=True)

In [ ]:
print(tau)

flat_samples = backend.get_chain(discard=int(tau[0]), flat=True)

In [ ]:
from IPython.display import display, Math
mcmc = []
for i in range(ndim):
    mcmc_i = np.percentile(flat_samples[:, i], [16, 50, 84])
    q = np.diff(mcmc_i)
    txt = "\mathrm{{{3}}} = {0:.3f}_{{-{1:.3f}}}^{{+{2:.3f}}}"
    txt = txt.format(mcmc_i[1], q[0], q[1], labels[i])
    display(Math(txt))
    mcmc.append(mcmc_i)

In [ ]:

fig = corner.corner(
    flat_samples, labels=labels, truths=[mcmc[i][1] for i in range(ndim)]
)
fig.savefig(source + '_'+ molecule+ "_mcmccorner.pdf", dpi=72, bbox_inches='tight')

## Step 5: Analysis of results
We load the MCMC results and make the best fit trajectories with it